## 1. Imports and settings

In [4]:
#numpy pandas scipy scikit-learn matplotlib folium joblib
import json, time, zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
import folium
import joblib
import sklearn

from scipy.spatial import cKDTree
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components
from scipy.stats import norm

from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import Matern, WhiteKernel
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             log_loss, confusion_matrix, ConfusionMatrixDisplay)

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

K_CAND = 10          # candidate roads kept per GPS observation
TRAIN_STRIDE = 5     # use every 5th GPS observation of a training trip for model fitting (speed)
FLAG_DIST_M = 50     # trip is flagged if its median GPS-to-reference-route distance exceeds this
CV_FOLDS = 5

# Only the three FastAPI handoff artifacts are saved here.
OUT = Path("pathsnap_output-1")
OUT.mkdir(exist_ok=True)
FEATURES = ["distance_m", "midpoint_distance_m", "segment_length_m", "speed_mps",
            "direction_difference", "distance_rank", "relative_distance_m"]

## 2. Locate the dataset
Keep the dataset ZIP next to this notebook (any `*.zip`) or set `ZIP_PATH`. If the data is already extracted into `pathsnap_data/`, the ZIP is not needed.

In [5]:
DATA_DIR = Path("Student folder")  # Change this to your extracted dataset folder if needed

def find_dirs(root):
    dirs = [p for p in root.rglob("*") if p.is_dir() and "__MACOSX" not in p.parts]

    train = [p for p in dirs if p.name.lower() == "train"]
    test = [p for p in dirs if "test" in p.name.lower() and "evaluator" not in p.name.lower()]

    return (train[0] if train else None), (test[0] if test else None)


TRAIN_DIR, STUDENT_TEST_DIR = find_dirs(DATA_DIR)

assert TRAIN_DIR is not None, "No 'train' folder found."
assert STUDENT_TEST_DIR is not None, "No student-test folder found."

print("Training folder    :", TRAIN_DIR)
print("Student-test folder:", STUDENT_TEST_DIR)


Training folder    : Student folder/train
Student-test folder: Student folder/student_test
